<a id="unit-u.b9230fdbcde4447cb907f4baca427b3d"></a>

<div>
  <div>8-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Бір анықтаманы ортақ қолдану, нәтижесін сақтау және қайта жазуды тексеру</div>
</div>

Екі топ 1-деңгей қалдырған тапсырыстардан «күндік тапсырысты» санап, әртүрлі сан алады. Бір анықтаманы келісіп, кәдімгі көрініс ретінде жариялайсыз, күндік нәтижесін асинхронды материалданған көріністе сақтайсыз, жаңартуы мен жолдарын тексересіз және `EXPLAIN` арқылы қай сұрауларға жауап беретінін оқисыз. Содан кейін жеке шағын кестелерде сақталған нәтиженің ескіріп, қуып жетуін, әр жазу жүргізетін синхронды материалданған көріністі және бір не екі күнді жаңартатын бөлімделген материалданған көріністі бақылайсыз. Шамамен 65 минут жоспарлаңыз.

**8-модуль: көріністер және материалданған көріністер** · [Сабақ](course.md) · [Тест](quiz08_views_and_materialized_views.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.2d6b4d31211c4125a94ccb67ecdabedf"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Алдымен сол дерекқорда 7-зертханалық жұмысты аяқтаңыз. Бұл жұмыс 7-зертханалық жұмыс қалдырған тапсырыстардың ағымдағы күйі мен тарихын — `orders_current` және `order_events` — оқиды, екеуін де өзгертпейді. Олар жоқ болса, баптау ұяшығы тоқтап, қай жұмысты аяқтау керегін айтады.

Жұмыс тек `_l2` жұрнағымен аяқталатын өз нысандарын қайта жасайды: `orders_view_l2` көрінісі, `orders_daily_mv_l2` материалданған көрінісі, жаттығуда `orders_region_mv_l2`. 5–8-бөлімдер жеке шағын кестелерде орындалады: `orders_refresh_view_l2` және `orders_refresh_mv_l2` бар `orders_refresh_source_l2`; `mv_sync_sum_l2` бар `mv_sync_source_l2`; `orders_region_partition_mv_l2` бар `orders_region_partition_l2` және `customer_region_mv_l2`. Жұмыстан кейін олар зертханалық дерекқорыңызда қалады. Бәрі курс сынақ ортасында орындалады; басқа контейнер қажет емес.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, normalized
from dw_course.ui import show_records, show_sql
from dw_course.warehouse import wait_for_mv

lab = connect_sandbox(module=8)

<a id="unit-u.d4057c1ca0f64118b297fd243937dfc2"></a>

<a id="1-two-answers-to-daily-orders"></a>

## 1. «Күндік тапсырысқа» екі жауап

Операция және қаржы топтары күндік тапсырыстарды сұрайды. `orders_current` әр тапсырысқа бір жол сақтайды, оның `event_time` мәні — берілген уақыты емес, жеткізілген, бас тартылған не ақшасы қайтарылған соңғы оқиғасының уақыты. `order_events` әр оқиғаны сақтайды; тапсырыстың 1-нұсқасы — оны жасаған оқиға.

Бірінші сұрау ағымдағы тапсырыстарды соңғы оқиғасының күнімен топтайды. Екіншісі оларды жасаған оқиғаның күнімен топтап, бас тартылғандарды тыс қалдырады. Әрқайсысы 1 қаңтарда қанша тапсырыс санайтынын болжап, ұяшықты орындаңыз.


In [ ]:
lab.sql("""
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_current
GROUP BY DATE(event_time)
ORDER BY order_date
""", title="Grouped by the date of each order's latest event")
lab.sql("""
SELECT DATE(e.event_time) AS order_date, COUNT(*) AS order_count, SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time)
ORDER BY order_date
""", title="Grouped by the date each order was created, without cancelled orders");

<a id="unit-u.98ab83dc4d804f8fa99aafb4d2f217a8"></a>

**Күтілетін нәтиже**

Екі кесте. Соңғы оқиға бойынша: 2026-01-01 күні 7 тапсырыс және 950.00, 2026-01-02 күні 4 тапсырыс және 560.00. Жасалу бойынша, бас тартылғансыз: 2026-01-01 күні 9 тапсырыс және 1200.00, 2026-01-02 күні 1 тапсырыс және 110.00.

Екі сұрау да орындалады, екеуінің SQL-і де орынды. Біріншісі 900001, 900002 және 900003 тапсырыстарын жеткізілген, бас тартылған және қайтарылған 2 қаңтарда санайды, бас тартылған 900002 мен оның 200.00 сомасын да санайды. Екіншісі бас тартылмаған әр тапсырысты берілген күнінде бір рет санайды. Курс екінші анықтаманы алады: тапсырыс күні — жасалған күні, бас тартылған тапсырыс саналмайды. Келесі қадам мұны барлығына бір рет жазып қояды.


<a id="unit-u.2c8280b035064360a00e6da2afd4237e"></a>

<a id="2-publish-the-definition-as-a-regular-view"></a>

## 2. Анықтаманы кәдімгі көрініс ретінде жариялау

Кәдімгі көрініс сұрауды атаумен сақтайды. `orders_view_l2` есеп үшін тапсырыс не екенін айтады: `order_date` — 1-нұсқа оқиғасының күні, бас тартылған тапсырыстар тыс қалады, есептер қолданатын бағандар көрсетіледі. Әр сұрау сақталған сұрауды ағымдағы кестелерде орындайды: көрініс жолдар сақтамайды. Ол қатынау рұқсатын да бермейді; 12-модуль рұқсат береді.

Қайта орындауға болу үшін ұяшық алдымен осы жұмыстың көріністерін жояды.


In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_mv_l2")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_daily_mv_l2")
lab.execute("DROP VIEW IF EXISTS orders_view_l2")
view_ddl = """CREATE VIEW orders_view_l2 AS
SELECT o.order_id,
       DATE(e.event_time) AS order_date,
       o.customer_id, o.status, o.order_amount, o.paid_amount, o.refund_amount, o.region
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'"""
show_sql("The view's definition", view_ddl)
lab.execute(view_ddl)
lab.sql("SELECT * FROM orders_view_l2 ORDER BY order_id", title="The orders that count, with their order date")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_view_l2"), [(10, "1310.00")],
       title="Ten orders, 1310.00: every order except the cancelled 900002")

<a id="unit-u.787472a50cf2496b93ada701027c11cb"></a>

**Күтілетін нәтиже**

Анықтама, содан кейін он жол: 900002-сіз 900001 және 900003–900011 тапсырыстары. Тоғызының `order_date` мәні 2026-01-01, 900011-дікі 2026-01-02; 900001 DELIVERED, 900003 REFUNDED, 150.00 төленген және 150.00 қайтарылған, басқалары CREATED. Жасыл белгі: он тапсырыс, 1310.00.

`order_amount` — алынған ақша емес, тапсырыстың салыққа дейінгі құны: 900003 ақшасы қайтарылса да, 150.00 сақтайды. Төлемдер мен қайтарымдар — өз көрсеткіштері, `paid_amount` және `refund_amount`.


<a id="unit-u.cec3b045433c41e7b8c8ae44bac5b2a7"></a>

<a id="3-store-the-daily-result-in-an-asynchronous-materialized-view"></a>

## 3. Күндік нәтижені асинхронды материалданған көріністе сақтау

Күндік қорытындыларды ашатын әр бақылау тақтасы тапсырыстар өзгергенше бірдей жауап алады. Асинхронды материалданған көрініс сол жауапты сақтайды: көрініс сияқты сұрауы, кесте сияқты жаңарту қайта есептейтін нәтижесі бар.

- `BUILD IMMEDIATE` көрініс жасалғанда алғашқы құруды бастайды.
- `REFRESH AUTO` Doris-ке қаншасын қайта есептеуді шешкізеді: қайсысы өзгергенін анықтай алса, өзгерген бөлімдер, әйтпесе бәрі.
- `ON MANUAL`: жаңартуды тек `REFRESH MATERIALIZED VIEW` операторы бастайды. Сұрау бастамайды.

Материалданған көрініс `orders_view_l2` кестелерімен, біріктіруімен және сүзгісімен бірдей. Құруы `CREATE` қайтқаннан кейін тапсырма ретінде орындалады, сондықтан ұяшық `mv_infos` құруды хабарлағанша күтіп, кейін сақталған жолдарды анықтамамен салыстырады.


In [ ]:
mv_ddl = """CREATE MATERIALIZED VIEW orders_daily_mv_l2
BUILD IMMEDIATE
REFRESH AUTO ON MANUAL
AS
SELECT DATE(e.event_time) AS order_date,
       COUNT(*) AS order_count,
       SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time)"""
show_sql("The materialized view's definition", mv_ddl)
lab.execute(mv_ddl)
mv_state = ("SELECT Name, State, RefreshState, SyncWithBaseTables FROM mv_infos('database'=%s) "
            "WHERE Name = 'orders_daily_mv_l2'")
wait_for_mv(lab, "orders_daily_mv_l2", lambda: lab.query(mv_state, (lab.database,)),
            lambda rows: bool(rows) and rows[0][2] == "SUCCESS", description="The first build of orders_daily_mv_l2")
lab.sql(mv_state, (lab.database,), title="The materialized view's state")
lab.sql("SELECT * FROM orders_daily_mv_l2 ORDER BY order_date", title="The stored daily result")
expect(lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount FROM orders_daily_mv_l2 ORDER BY order_date"),
       [("2026-01-01", 9, "1200.00"), ("2026-01-02", 1, "110.00")], title="The stored result matches the definition")

<a id="unit-u.5165dcf0f2ae4c37a3746b3abcd6d7d8"></a>

**Күтілетін нәтиже**

Анықтама, бірнеше секундтан кейін жасыл болатын барыс карточкасы, содан кейін күй: `State` NORMAL, `RefreshState` SUCCESS, `SyncWithBaseTables` 1. Сақталған нәтижеде екі жол бар: 2026-01-01 күні 9 тапсырыс және 1200.00, 2026-01-02 күні 1 тапсырыс және 110.00, тексеру жасыл.

Үш күй бағаны әртүрлі сұраққа жауап береді: `State` материалданған көріністі мүлде қолдануға бола ма, `RefreshState` соңғы жаңарту сәтті ме, `SyncWithBaseTables` нәтиже негізгі кестелерге әлі сәйкес пе. Сәтті `CREATE` бұлардың ешқайсысын айтпайды; тек анықтаманы қабылдады.


<a id="unit-u.141030ebe7f0436e9f7e38863e80829d"></a>

<a id="4-read-in-explain-whether-a-query-used-the-materialized-view"></a>

## 4. EXPLAIN арқылы сұраудың материалданған көріністі қолданғанын оқу

Бақылау тақталары материалданған көріністі емес, `orders_view_l2` сұрайды. Сұрау материалданған көрініс сақтайтынды сұраса, Doris негізгі кестелердің орнына сақталған жолдарды оқитындай қайта жаза алады: мөлдір қайта жазу. `EXPLAIN` сұрауды орындамай жоспарды көрсетеді: әр сканерлеу қай кестені не материалданған көріністі оқитынын және Doris сынаған материалданған көріністерді таңдалған, жарамды, бірақ таңдалмаған не себебімен жарамсыз деп тізеді.

Ұяшық көріністің үш сұрауын түсіндіреді: күндік қорытындылар, күн мен аймақ қорытындылары және тек 1 қаңтар қорытындысы. Материалданған көрініс қайсысына жауап бере алатынын болжаңыз.


In [ ]:
daily_sql = """SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 GROUP BY order_date"""
scans, views = lab.explain_scan(daily_sql, title="The daily totals of the view")
expect([scan["reads"] for scan in scans], ["orders_daily_mv_l2"], title="The daily query reads orders_daily_mv_l2")
region_sql = """SELECT order_date, region, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 GROUP BY order_date, region"""
scans, views = lab.explain_scan(region_sql, title="The totals per day and region")
expect(sorted(scan["reads"] for scan in scans), ["order_events", "orders_current"],
       title="The query per region reads the base tables")
one_day_sql = """SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 WHERE order_date = '2026-01-01' GROUP BY order_date"""
scans, views = lab.explain_scan(one_day_sql, title="The totals of January 1")
expect(sorted(scan["reads"] for scan in scans), ["order_events", "orders_current"],
       title="The query of one day reads the base tables too")
lab.sql(daily_sql + " ORDER BY order_date", title="The daily totals, as a dashboard sees them");

<a id="unit-u.6559cb6b182f453da58480afeecfe6df"></a>

**Күтілетін нәтиже**

Үш жоспар, әрқайсысында жасыл белгі.

- Күндік қорытындылар: таңдалған деп тізілген `orders_daily_mv_l2` оқитын бір сканерлеу.
- Күн мен аймақ қорытындылары: `orders_current` және `order_events` екі сканерлеуі, `orders_daily_mv_l2` жарамсыз: `View struct info is invalid, View dimensions doesn't not cover the query dimensions`.
- 1 қаңтар қорытындысы: тағы екі сканерлеу, `orders_daily_mv_l2` жарамсыз: `View struct info is invalid, Rewrite compensate predicate by view fail`. Толық жоспарда `order_events` сканерлеуінің предикаты `event_time >= '2026-01-01 00:00:00' AND event_time <= '2026-01-01 23:59:59'`.

Содан кейін күндік қорытындылар: 9 тапсырыс және 1200.00, 1 тапсырыс және 110.00.

Күндік сұрау тек көріністі атайды, ал Doris сақталған нәтижені оқыды. Аймақ сұрауы оны қолдана алмайды: сақталған жолдарда аймақ жоқ, қайта жазу материалданған көрініс сақтамаған бағанды қайтара алмайды. Бір күн сұрауы таңғалдырады: сақталған жолдарда `order_date` бар, бірақ Doris `DATE(event_time) = '2026-01-01'` шартын материалданған көрініс сақтамайтын `event_time` бағанының ауқымына айналдырды, сондықтан сүзгіні сақталған жолдарға қолдана алмады (Doris 4.1.3). Қай сұраулар қайта жазылғанын тек `EXPLAIN` көрсетеді; еш жоспар жауаптың дұрыстығын айтпайды: 3-қадам сақталған жолдарды анықтамамен салыстырды.
Жоспардың `tablets` саны нәтиже жолдарының санын емес, физикалық сақтауды сипаттайды. Бұл шағын MV Doris-тің әдепкі таратуын қолданады, сондықтан жоғарыдағы екі күндік жолды ғана сақтаса да, осы сынақ ортасында `tablets=10/10` бере алады. Tablet саны мен сұрау нәтижесін бөлек оқыңыз; көбірек tablet көбірек күндік топ не жылдамырақ сұрау деген сөз емес.


<a id="unit-u.689df6dabdf24c2490de5129f47ab316"></a>

<a id="5-watch-a-stored-result-go-stale-then-catch-up"></a>

## 5. Сақталған нәтиженің ескіруін, содан кейін қуып жетуін бақылау

Кейінгі жұмыстар 1-деңгей кестелерін оқиды, сондықтан бұл қадам 100.00 сомалы бір тапсырысы бар жеке шағын `orders_refresh_source_l2` кестесіне жазады. Оған бірдей күндік сұрауы бар кәдімгі және материалданған көрініс беріледі. Содан кейін 50.00 сомалы екінші тапсырыс келеді. Ұяшықты орындамас бұрын екі нәтижені болжаңыз: кәдімгі көрініс сұрауын қазір орындайды; `ON MANUAL` материалданған көрініс әлі соңғы жаңарту нәтижесін сақтайды.


In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_refresh_mv_l2")
lab.execute("DROP VIEW IF EXISTS orders_refresh_view_l2")
lab.execute("DROP TABLE IF EXISTS orders_refresh_source_l2")
lab.execute("""
CREATE TABLE orders_refresh_source_l2 (
    event_time DATETIME NOT NULL,
    order_id BIGINT NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(event_time, order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_refresh_source_l2", ["event_time", "order_id", "order_amount"], [("2026-01-01 12:00:00", 999001, 100)])
lab.execute("""
CREATE VIEW orders_refresh_view_l2 AS
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_refresh_source_l2 GROUP BY DATE(event_time)
""")
lab.execute("""
CREATE MATERIALIZED VIEW orders_refresh_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
AS SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_refresh_source_l2 GROUP BY DATE(event_time)
""")
wait_for_mv(lab, "orders_refresh_mv_l2",
            lambda: normalized(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2")),
            lambda rows: rows == [[1, "100.00"]], description="The first build of orders_refresh_mv_l2")
lab.insert("orders_refresh_source_l2", ["event_time", "order_id", "order_amount"], [("2026-01-01 13:00:00", 999002, 50)])
contrast_sql = """
SELECT 'regular view' AS result_source, order_count, gross_amount FROM orders_refresh_view_l2
UNION ALL
SELECT 'stored MV', order_count, gross_amount FROM orders_refresh_mv_l2
ORDER BY result_source
"""
lab.sql(contrast_sql, title="After the write, the regular view and the stored result disagree")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_view_l2"), [(2, "150.00")],
       title="The regular view includes the new order")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2"), [(1, "100.00")],
       title="The materialized view still holds its last result")

<a id="unit-u.a792a928f9304181b77f7175ae56741d"></a>

**Күтілетін нәтиже**

Алғашқы құрудың барыс карточкасы, содан кейін екі жолдық кесте: 2 тапсырыс және 150.00 бар `regular view`, 1 тапсырыс және 100.00 бар `stored MV`, екі жасыл белгі. Материалданған көрініс алғашқы құру есептеген нәтижені әлі сақтайды.


<a id="unit-u.19fd6ba525ec4fc09c7522ef5188f2f2"></a>

Енді оны жаңартыңыз. `REFRESH ... COMPLETE` бүкіл нәтижені қайта есептейді; оператор тапсырма жіберілгенде қайтады, сондықтан ұяшық салыстырмас бұрын жаңа нәтижені күтеді. Кәдімгі көрініске жаңарту қажет емес: оның сақталған нәтижесі жоқ.


In [ ]:
lab.execute("REFRESH MATERIALIZED VIEW orders_refresh_mv_l2 COMPLETE")
wait_for_mv(lab, "orders_refresh_mv_l2",
            lambda: normalized(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2")),
            lambda rows: rows == [[2, "150.00"]], description="The manual refresh of orders_refresh_mv_l2")
lab.sql(contrast_sql, title="After the refresh, both results agree")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2"), [(2, "150.00")],
       title="The stored result caught up after the refresh")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_current"), [(11, "1510.00")],
       title="Lab 7's current orders are unchanged")

<a id="unit-u.c49cefb60f754c64b1c363337553bb4e"></a>

**Күтілетін нәтиже**

Барыс карточкасы, содан кейін екі нәтиже де 2 тапсырыс және 150.00 көрсетеді, екі жасыл белгі: сақталған нәтиже қуып жетті, ал 7-зертханалық жұмыстың `orders_current` кестесінде әлі 11 тапсырыс және 1510.00 бар.

Жаңарту операторының сәттілігі, бақылау тақтасына жеткілікті өзекті нәтиже және нәтижені оқуға қайта жазылған сұрау — үш бөлек тексеру: 4-қадам қайта жазуды, бұл қадам жаңартуды тексерді.


<a id="unit-u.a1391fee847b4dd3aae4a744b32c10cb"></a>

<a id="6-a-synchronous-materialized-view-maintained-by-every-write"></a>

## 6. Әр жазу жүргізетін синхронды материалданған көрініс

Синхронды материалданған көрініс бір кестеге тиесілі және кестеге әр жүктеу оны да сол транзакцияда жаңартады: орындайтын бөлек жаңарту жоқ. Жеке `mv_sync_source_l2` Duplicate Key кестесінде күн бойынша соманы қосатын көрініс жасаңыз. Алғашқы құруы — фондық тапсырма, сондықтан ұяшық `CREATE` алдындағы уақытты тіркеп, `SHOW ALTER TABLE MATERIALIZED VIEW` осы құруды `FINISHED` деп көрсеткенше күтеді. Содан кейін екінші тапсырысты жүктеп, негізгі кестенің күндік қосындысын түсіндіреді.

Болжаңыз: күндік қосындыға кіру үшін екінші тапсырысқа жаңарту қажет пе?


In [ ]:
from dw_course.streaming import wait_for
import pymysql

lab.execute("DROP TABLE IF EXISTS mv_sync_source_l2")
lab.execute("""
CREATE TABLE mv_sync_source_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("mv_sync_source_l2", ["order_date", "order_id", "amount"], [("2026-02-01", 8101, 100)])
created_after = str(lab.query("SELECT NOW()")[0][0])
lab.execute("""
CREATE MATERIALIZED VIEW mv_sync_sum_l2 AS
SELECT order_date AS mv_day, SUM(amount) AS mv_amount
FROM mv_sync_source_l2 GROUP BY order_date
""")
build_job = (f"SHOW ALTER TABLE MATERIALIZED VIEW FROM {lab.database} WHERE TableName = 'mv_sync_source_l2' "
             f"AND RollupIndexName = 'mv_sync_sum_l2' AND CreateTime >= '{created_after}'")

def build_state():
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute(build_job)
        jobs = cursor.fetchall()
    if jobs and jobs[-1]["State"] == "CANCELLED":
        raise RuntimeError(f"The build of mv_sync_sum_l2 was cancelled: {jobs[-1]['Msg']}")
    return jobs[-1]["State"] if jobs else None

wait_for(build_state, lambda state: state == "FINISHED", description="The first build of mv_sync_sum_l2", timeout=120)
lab.sql(build_job, title="This build of mv_sync_sum_l2",
        columns=["JobId", "TableName", "RollupIndexName", "CreateTime", "FinishTime", "State"])
lab.insert("mv_sync_source_l2", ["order_date", "order_id", "amount"], [("2026-02-01", 8102, 50)])
scans, views = lab.explain_scan("SELECT order_date, SUM(amount) FROM mv_sync_source_l2 GROUP BY order_date",
                                title="A daily sum of the base table")
expect([scan["reads"] for scan in scans], ["mv_sync_sum_l2"], title="The plan reads mv_sync_sum_l2, not the table's rows")
expect(lab.query("SELECT SUM(amount) FROM mv_sync_source_l2"), [("150.00",)], title="The second order counts without a refresh")

<a id="unit-u.919794d835514645bb4d2bbb50ef2be6"></a>

**Күтілетін нәтиже**

Барыс карточкасы, осы орындаудың `FINISHED` құру тапсырмасы, содан кейін жоспар: `mv_sync_sum_l2` оқитын `mv_sync_source_l2` бір сканерлеуі, материалданған көрініс таңдалған деп тізілген, екі жасыл белгі — жоспар `mv_sync_sum_l2` оқиды, күндік қосынды 150.00.

Екінші тапсырыс жүктеуі бекітілген бойда `mv_sync_sum_l2` ішінде болды, себебі жүктеу оны жаңартты. Шығын жазуларға ауысты, олар енді жолдардың екі жиынын жүргізеді; кестедегі әр синхронды материалданған көрініс жүктеулерге көбірек жұмыс істетеді.


<a id="unit-u.adff09564b9748f386e7812b7471badd"></a>

<a id="7-refresh-only-the-changed-partition"></a>

## 7. Тек өзгерген бөлімді жаңарту

Бөлімделген асинхронды материалданған көрініс дереккөзінің бір бөлімі ғана өзгерсе, бір бөлімді жаңарта алады. `orders_region_partition_l2` екі күндік бөлімге ие және бөлімделмеген шағын `customer_region_mv_l2` тұтынушы кестесімен біріктіріледі; `PARTITION BY (order_date)` материалданған көрінісі тапсырыстарды күн мен аймақ бойынша топтайды, сондықтан әр бөлімі бір факт күніне сәйкес келеді.

Болжаңыз: 2 ақпанның жаңа тапсырысы 2 ақпан бөлімін жаңартып, 1 ақпанды қалдыруы керек. Ұяшық әр бөлімнің бұрынғы және кейінгі `PartitionId` мәнін салыстырады: қайта жарияланған бөлім `VisibleVersion` 2 болып қалса да, жаңа ID алады. Содан кейін жаңарту тапсырмасының жазбасын оқиды.


In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_partition_mv_l2")
lab.execute("DROP TABLE IF EXISTS orders_region_partition_l2")
lab.execute("DROP TABLE IF EXISTS customer_region_mv_l2")
lab.execute("""
CREATE TABLE orders_region_partition_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p20260201 VALUES [("2026-02-01"), ("2026-02-02")),
    PARTITION p20260202 VALUES [("2026-02-02"), ("2026-02-03"))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
CREATE TABLE customer_region_mv_l2 (customer_id BIGINT NOT NULL, region VARCHAR(16) NOT NULL)
UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("customer_region_mv_l2", ["customer_id", "region"], [(1,"EAST"),(2,"WEST")])
lab.insert("orders_region_partition_l2", ["order_date","order_id","customer_id","amount"],
           [("2026-02-01",8201,1,100),("2026-02-02",8202,2,200)])
lab.execute("""
CREATE MATERIALIZED VIEW orders_region_partition_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
PARTITION BY (order_date)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1", "refresh_partition_num"="10")
AS SELECT o.order_date, c.region, COUNT(*) AS order_count, SUM(o.amount) AS gross_amount
FROM orders_region_partition_l2 o JOIN customer_region_mv_l2 c ON o.customer_id=c.customer_id
GROUP BY o.order_date, c.region
""")
region_result = "SELECT CAST(order_date AS STRING), region, order_count, gross_amount FROM orders_region_partition_mv_l2 ORDER BY order_date, region"
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","EAST",1,"100.00"],["2026-02-02","WEST",1,"200.00"]],
            description="The first build of orders_region_partition_mv_l2", timeout=120)

def mv_partition_publications():
    frame = lab.sql("SHOW PARTITIONS FROM orders_region_partition_mv_l2", title="The materialized view's partitions",
                    columns=["PartitionId", "PartitionName", "VisibleVersion", "VisibleVersionTime"])
    return {row.PartitionName: (row.PartitionId, row.VisibleVersion) for row in frame.itertuples()}

versions_before = mv_partition_publications()
lab.insert("orders_region_partition_l2", ["order_date","order_id","customer_id","amount"], [("2026-02-02",8203,2,50)])
lab.execute("REFRESH MATERIALIZED VIEW orders_region_partition_mv_l2 AUTO")
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","EAST",1,"100.00"],["2026-02-02","WEST",2,"250.00"]],
            description="The refresh after the new order on February 2", timeout=120)
versions_after_fact = mv_partition_publications()
expect(sum(versions_after_fact[name] != version for name, version in versions_before.items()), 1,
       title="Exactly one partition was published again")
refresh_tasks = ("SELECT TaskId, Status, RefreshMode, NeedRefreshPartitions FROM tasks('type'='mv') "
                 "WHERE MvName = 'orders_region_partition_mv_l2' AND MvDatabaseName = %s ORDER BY CreateTime DESC LIMIT 3")
lab.sql(refresh_tasks, (lab.database,), title="The refresh tasks, newest first");

<a id="unit-u.2a3732c8c96f4e6d9224d6ced42724de"></a>

**Күтілетін нәтиже**

Барыс карточкасы және материалданған көріністің екі бөлімі; жаңа тапсырыстан кейін екінші барыс карточкасы және бөлімдер қайта көрсетіледі, мұнда `p_20260202_20260203` жаңа `PartitionId` алады, `p_20260201_20260202` бұрынғысын сақтайды. Жасыл белгі: дәл бір бөлім қайта жарияланды. Тапсырмалар жаңасынан бастап: `RefreshMode` PARTIAL және `NeedRefreshPartitions` `["p_20260202_20260203"]` бар жаңарту, содан кейін екі бөлімі бар алғашқы құру.


<a id="unit-u.bcaa85c4148a40d8a54658a3e39cdf6e"></a>

<a id="8-a-dimension-change-can-widen-the-refresh"></a>

## 8. Өлшем өзгерісі жаңарту аясын кеңейте алады

Енді 1-тұтынушының аймағын EAST-тен NORTH-қа өзгертіңіз. Тұтынушы кестесі бөлімделмеген, сондықтан өзгерісті бір күнге сәйкестендіретін ештеңе жоқ. `REFRESH ... AUTO` материалданған көріністің әр бөлімі өзгеруі мүмкін деп қабылдауы керек, бұл жаңарту сандары өзгермегенін қоса екі бөлімді де жариялайды.

Жаңа `PartitionId` бөлімнің қайта жарияланғанын көрсетеді, оған қанша жұмыс кеткенін емес. Тапсырма жазбасы қай бөлімдер қайта есептелгенін, ал Query Profile істелген жұмысты көрсетер еді.


In [ ]:
lab.insert("customer_region_mv_l2", ["customer_id","region"], [(1,"NORTH")])
lab.execute("REFRESH MATERIALIZED VIEW orders_region_partition_mv_l2 AUTO")
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","NORTH",1,"100.00"],["2026-02-02","WEST",2,"250.00"]],
            description="The refresh after the region change of customer 1", timeout=120)
versions_after_dimension = mv_partition_publications()
expect(sum(versions_after_dimension[name] != version for name, version in versions_after_fact.items()), 2,
       title="The dimension change published both partitions")
show_records("Refresh comparison", [
    {"change":"one fact date", "partitions_changed":1, "other_day_amount":"100.00"},
    {"change":"unpartitioned dimension", "partitions_changed":2, "other_day_amount":"250.00"},
])
lab.sql(refresh_tasks, (lab.database,), title="The refresh tasks, newest first");

<a id="unit-u.d863f13fc31b4142860793f428526e60"></a>

**Күтілетін нәтиже**

Барыс карточкасы, содан кейін бөлімдер: екеуінде де жаңа `PartitionId`, тексеру жасыл. Салыстыру: бір факт күні бір бөлімді жариялады; өлшем өзгерісі екеуін де жариялады, 2 ақпанның 250.00 сомасы өзгермесе де. Ең жаңа тапсырмада `RefreshMode` COMPLETE және `NeedRefreshPartitions` ішінде екі бөлім де бар.

Жаңарту аясы Doris нені материалданған көріністің бөлімдеріне сәйкестендіре алатынына байланысты. Факт бөлімі бір күнге сәйкес; бөлімделмеген кестедегі өзгеріс ешбір күнге сәйкес емес, сондықтан бәрі қайта есептеледі.


<a id="unit-u.aea34271e1db4685bbe9ec3f491e2ba1"></a>

<a id="9-practice-daily-orders-per-region"></a>

## 9. Жаттығу: аймақ бойынша күндік тапсырыстар

Аймақтық бақылау тақтасы әр күн мен аймақ тапсырыстарын сұрайды. 4-қадам `orders_daily_mv_l2` жауап бере алмайтынын көрсетті: жолдарында аймақ жоқ. `orders_view_l2` кестелерімен, біріктіруімен және сүзгісімен бақылау тақтасына қажет түйіршіктілікте екінші `orders_region_mv_l2` материалданған көрінісін жасаңыз. Алғашқы құруын күтіп, жолдарын көріністің күн мен аймақ қорытындыларымен салыстырыңыз, содан кейін 4-қадамның аймақ сұрауын қайта түсіндіріңіз. `orders_daily_mv_l2` өзгеріссіз қалсын: тұтынушылары әлі оны оқиды.


In [ ]:
# Your code here

<a id="unit-u.1a4b62a5c74242cd84400abe52340196"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_mv_l2")
lab.execute("""CREATE MATERIALIZED VIEW orders_region_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
AS
SELECT DATE(e.event_time) AS order_date, o.region,
       COUNT(*) AS order_count, SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time), o.region""")
stored = "SELECT CAST(order_date AS STRING) AS order_date, region, order_count, gross_amount FROM orders_region_mv_l2 ORDER BY order_date, region"
from_view = """SELECT CAST(order_date AS STRING), region, COUNT(*), SUM(order_amount)
FROM orders_view_l2 GROUP BY order_date, region ORDER BY order_date, region"""
wait_for_mv(lab, "orders_region_mv_l2", lambda: normalized(lab.query(stored)), lambda rows: len(rows) == 3,
            description="The first build of orders_region_mv_l2")
lab.sql(stored, title="The stored totals per day and region")
expect(lab.query(stored), lab.query(from_view), title="The stored rows equal the view's totals")
scans, views = lab.explain_scan(region_sql, title="The totals per day and region, again")
expect([scan["reads"] for scan in scans], ["orders_region_mv_l2"], title="The query per region now reads orders_region_mv_l2")

<a id="unit-u.2106b2fa3af446b48be66d49613a070f"></a>

**Күтілетін нәтиже**

Барыс карточкасы, содан кейін сақталған үш жол: (2026-01-01, EAST, 5, 720.00), (2026-01-01, WEST, 4, 480.00), (2026-01-02, EAST, 1, 110.00), жасыл белгі: олар көрініс қорытындыларына тең. Жоспар таңдалған деп тізілген `orders_region_mv_l2` оқиды, `orders_daily_mv_l2` бұл сұрауға әлі жарамсыз; соңғы тексеру жасыл.

Жаңа материалданған көрініс `orders_daily_mv_l2` кестесіне қарағанда егжей-тегжейлі түйіршіктілік сақтайды, сондықтан жолдары көбірек және жоспарлап, тексеретін тағы бір жаңарту қосады.


<a id="unit-u.648b4b4b573846fd9918b92bd4baa3c4"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- екі орынды сұрау 1 қаңтарда неге 7 және 9 тапсырыс санағанын және `orders_view_l2` қай анықтаманы жариялайтынын айта аласыз;
- `orders_daily_mv_l2` `RefreshState` SUCCESS хабарлайды, жолдары анықтамаға тең және `EXPLAIN` арқылы қай сұрауларға жауап беретінін, басқалары неге қолдана алмайтынын оқи аласыз;
- қолмен жаңартылатын нәтиженің ескіріп, қуып жетуін, жүктеу жаңартатын синхронды материалданған көріністі және бір бөлім жаңартуын екі бөлім жаңартуымен салыстырып көрдіңіз.


<a id="unit-u.79bdd230094148cf8961afa736ab0ab7"></a>

<a id="next"></a>

## Келесі қадам

[8-тест: көріністер және материалданған көріністер](quiz08_views_and_materialized_views.ipynb) файлын ашыңыз, содан кейін [9-модуль: деректер қоймасын модельдеу және біріктірулер](../module09-modeling-and-joins/course.md) бөліміне өтіңіз.
